In [1]:
import os
from dotenv import load_dotenv
import json

load_dotenv()

from params.paths import DATA_DIR
from dbio.representative_db import connect_db, iterate_all_persons, get_election_result_by_person_id


OUTPUT_DIR = os.path.join(DATA_DIR, "election_history")
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [5]:
conn = connect_db(
	dbname="kokkaidoc",
	user="postgres",
	password=os.getenv("PSQL_DATABASE_PASSWORD"),
	host="localhost",
	port="5432"
)

with conn.cursor() as cur:
	for person in iterate_all_persons(cur):
		election_results = get_election_result_by_person_id(cur, person.person_id)
		person_id = str(person.person_id)
		person_file_path  = os.path.join(OUTPUT_DIR, f"{person_id}.jsonl")
		with open(person_file_path, "w") as f:
			for election_result in election_results:
				f.write(json.dumps(election_result.raw_json, ensure_ascii=False) + "\n")


	
